# Caracal Bench CyberGym - GPU T4 x2 (agent loop, sem container runtime)

Roda CyberGym sample (3 arvo tasks) em GPU T4 x2 **sem Docker daemon nem udocker**.
Pull Docker image layers direto via Docker Registry HTTP API v2 (requests + tarfile)
+ subprocess.run pra executar /arvo script com paths reescritos.

Bench que Claude Mythos hit **83.1%** no full 1507 com 50-turn agent loop.
Aqui rodamos 3 arvo com max_turns=5 (cabe em ~1h GPU).

**Score deterministico**: PoC bytes que crashAM vul + NAO crashAM fix = pass.

**Baselines publicados (full 1507)**:
- Mythos 83.1, Opus 4.8 67, Sonnet 4.6 65, GPT-5 22, Gemini 2.5 Flash 4.8, DeepSeek-V3 3.6, Qwen3-235B 2.7, o4-mini 2.5

Settings: GPU T4 x2 (machine_shape NvidiaTeslaT4) + Internet ON + Persistence.

Estimado: 3 tasks × 5 turns × 2 modelos × ~30s/turn + 6 image pulls (~5min) ≈ 30-45min total.

In [ ]:
BENCH = "cybergym"
CHECKPOINT_DATASET = "pedroafonso2/caracal-base-3b-s01"
OUTPUT_DATASET = "pedroafonso2/caracal-bench-cybergym-s01"
print(f"bench={BENCH} checkpoint={CHECKPOINT_DATASET}")

In [ ]:
!pip install -q 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' kaggle 'requests>=2.31.0'

In [ ]:
import os
import subprocess

if not os.path.exists("/kaggle/working/caracal-1"):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            "dev",
            "https://github.com/iterate-labs-ai/caracal-1.git",
            "/kaggle/working/caracal-1",
        ],
        check=True,
    )
os.chdir("/kaggle/working/caracal-1")
rev = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"cloned, HEAD={rev}", flush=True)

In [ ]:
import torch

print(f"CUDA: {torch.cuda.is_available()}, n_gpu: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  gpu{i}: {torch.cuda.get_device_name(i)}")

In [ ]:
import subprocess

ckpt_dir = "/kaggle/working/ckpt"
subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        CHECKPOINT_DATASET,
        "-p",
        ckpt_dir,
        "--unzip",
        "--force",
    ],
    check=True,
)
subprocess.run(["ls", "-la", ckpt_dir], check=True)

In [ ]:
import subprocess
import sys

MAX_TURNS = 5
TASKS = ["arvo:47101", "arvo:3938", "arvo:24993"]

subprocess.run(
    [
        sys.executable,
        "-u",
        "eval/run_cybergym_local.py",
        "--adapter",
        ckpt_dir,
        "--out",
        "/kaggle/working/bench-adapter-cybergym.json",
        "--max-turns",
        str(MAX_TURNS),
        "--cache-dir",
        "/kaggle/working/cybergym-cache",
        "--tasks",
        *TASKS,
    ],
    check=True,
)

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-u",
        "eval/run_cybergym_local.py",
        "--out",
        "/kaggle/working/bench-base-cybergym.json",
        "--max-turns",
        str(MAX_TURNS),
        "--cache-dir",
        "/kaggle/working/cybergym-cache",
        "--tasks",
        *TASKS,
    ],
    check=True,
)

In [ ]:
import json
from pathlib import Path

ad = json.loads(Path("/kaggle/working/bench-adapter-cybergym.json").read_text())
ba = json.loads(Path("/kaggle/working/bench-base-cybergym.json").read_text())

comparison = {
    "caracal_adapter_pass_at_1": ad.get("pass_at_1"),
    "qwen_base_pass_at_1": ba.get("pass_at_1"),
    "delta": ad.get("pass_at_1", 0) - ba.get("pass_at_1", 0),
    "mythos_full_1507_pass_at_1": 0.831,
    "caracal_n_pass": ad.get("n_pass"),
    "base_n_pass": ba.get("n_pass"),
    "n_total": ad.get("n_total"),
}

diff = {
    "checkpoint_dataset": CHECKPOINT_DATASET,
    "bench": BENCH,
    "adapter": ad,
    "base": ba,
    "comparison": comparison,
}
pub_dir = Path("/kaggle/working/bench-published")
pub_dir.mkdir(parents=True, exist_ok=True)
(pub_dir / f"bench_{BENCH}_vs_base.json").write_text(json.dumps(diff, indent=2))
print(json.dumps(comparison, indent=2))

In [ ]:
import json
import subprocess

metadata = {
    "title": f"Caracal Bench {BENCH} s01",
    "id": OUTPUT_DATASET,
    "licenses": [{"name": "Apache-2.0"}],
}
(pub_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(pub_dir), "--public"],
    capture_output=True,
    text=True,
    check=False,
)
print(r.stdout, r.stderr)
if r.returncode != 0:
    subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(pub_dir), "-m", f"bench {BENCH}"], check=True
    )
print(f"Published -> {OUTPUT_DATASET}")